In [ ]:
import stim

In [ ]:
def make_noisy_circuit(circuit, noise = 0.001):
    noisy_circuit = stim.Circuit()

    for instruction in circuit:
        if instruction.name in ["CX", "CZ"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE2", instruction.targets_copy(), noise)
        elif instruction.name in ["M", "MX"]:
            noisy_circuit.append(instruction.name, instruction.targets_copy(), noise)
        elif instruction.name in ["R", "RX"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE1", instruction.targets_copy(), noise)
        elif instruction.name in ["QUBIT_COORDS", "DETECTOR", "TICK", "OBSERVABLE_INCLUDE"]:
            noisy_circuit.append(instruction)
        else:
            raise NotImplementedError(f"Incomplete noisification : {instruction.name}")

    noisy_circuit.compile_detector_sampler()
    noisy_circuit.compile_sampler()

    return noisy_circuit

In [ ]:
circuit = make_noisy_circuit(stim.Circuit().from_file("../assets/tqec-extended-stabilizers-detectors.stim"))
# What is the following value supposed to be ?
errors = circuit.shortest_graphlike_error(canonicalize_circuit_errors=True)
print(f"Length of shortest graph-like error : {len(errors)}")
for error in errors:
    print(str(error).split('\n')[3][8:])

In [ ]:
import sinter
from typing import List

tasks = [
    sinter.Task(
        circuit=make_noisy_circuit(
            stim.Circuit().from_file("../assets/tqec-extended-stabilizers-detectors.stim"), noise=noise
        ),
        json_metadata={'d': d, 'p': noise},
    )
    for d in [5]
    for noise in [0.0001, 0.001, 0.003125, 0.00625, 0.0125, 0.025, 0.05, 0.08, 0.1]
]

collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=['pymatching'],
    max_shots=100_000_000,
    max_errors=500,
)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: stats.json_metadata['d'],
)
ax.set_ylim(1e-8, 1e-0)
ax.set_xlim(9e-4, 1.2e-1)
ax.loglog()
ax.set_title("Proposed Spatial Junction [open ports]")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()
fig.set_dpi(120)